
# Model Context Protocol (MCP) 

The goal is to build a simple MCP server and a Python client that communicates with it over STDIO. The tasks are as follows:

1. **Server (`server.py`)**  
   - Create an MCP server named `"Demo"`.  
   - Add a tool `add(a: int, b: int) -> int` that returns the sum of two integers.  
   - Add a resource at the URI template `greeting://{name}` that returns the greeting string `"Hello, {name}!"`.  
   - Start the server loop using STDIO.

2. **Client (`client.py`)**  
   - Use the MCP CLI to spawn the server via STDIO.  
   - Create a `ClientSession`.  
   - List available resources and tools, printing their names.  
   - Read the resource `greeting://hello` and print its result.  
   - Call the tool `add` with arguments `a=1` and `b=7`, printing the result.

The following cells provide the Python code for `server.py` and `client.py`. You can run these scripts from a terminal or adapt them as needed.


## server.py

In [ ]:

# server.py
from mcp.server.fastmcp import FastMCP

# Create an MCP server with a name
mcp = FastMCP("Demo")

# Define a tool that adds two integers
@mcp.tool()
def add(a: int, b: int) -> int:
    '''Return the sum of a and b.'''
    return a + b

# Define a resource that returns a greeting
@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    '''Return a greeting for the provided name.'''
    return f"Hello, {name}!"

if __name__ == "__main__":
    # Start the server loop over STDIO
    mcp.run_stdio()


## client.py

In [ ]:

# client.py
import asyncio
from mcp import ClientSession
from mcp.client.stdio import stdio_client, StdioServerParameters

# Parameters to spawn the server via the mcp CLI
server_params = StdioServerParameters(
    command="mcp",
    args=["run", "server.py"],
    env=None
)

async def run():
    # Spawn the server and connect via STDIO
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialize the session
            await session.initialize()

            # List and print available resources
            resources = await session.list_resources()
            print("Resources:", [r["uri"] for r in resources])

            # List and print available tools
            tools = await session.list_tools()
            print("Tools:", [t["name"] for t in tools])

            # Read the greeting resource
            greeting = await session.read_resource("greeting://hello")
            print("Greeting:", greeting)

            # Call the add tool with a=1, b=7
            result = await session.call_tool("add", {"a": 1, "b": 7})
            print("add(1, 7) result:", result)

if __name__ == "__main__":
    asyncio.run(run())


### Expected Output

When you run the client script (after ensuring the server code is available and using the MCP CLI to spawn the server), you should see output similar to:

```
Resources: ['greeting://{name}']
Tools: ['add']
Greeting: Hello, hello!
add(1, 7) result: 8
```

This output confirms that the client successfully listed the resources and tools provided by the server, read the greeting resource, and invoked the `add` tool.
